# Homework 2 — Vector Search

LLM Zoomcamp 2026 · Cohort

Turning text into vectors and searching by meaning — not by keywords.

**Knowledge base:** 72 lesson markdown pages from the course repo at commit `8c1834d`  
**Embedding model:** `Xenova/all-MiniLM-L6-v2` via ONNX Runtime (no PyTorch needed)

## Setup

We use two helper scripts copied from the course repo (`02-vector-search/embed/`):
- `download.py` — fetches the ONNX model from HuggingFace the first time
- `embedder.py` — wraps ONNX Runtime to produce 384-dim normalized vectors

**Why ONNX instead of sentence-transformers?**  
The ONNX runtime produces identical vectors but skips PyTorch entirely — the install is ~30x smaller and works on any machine, including a basic Codespace.

In [ ]:
# Download the ONNX model if not already present (runs fast if model exists)
from download import download
download("Xenova/all-MiniLM-L6-v2")

In [1]:
import sys
import numpy as np
import minsearch
from gitsource import GithubRepositoryDataReader, chunk_documents

# embedder.py lives in the same directory as this notebook
sys.path.insert(0, ".")
from embedder import Embedder

## Loading the data

Same 72 lesson pages as homework 1. We pin to commit `8c1834d` for reproducibility.
Each document has a `filename` and a `content` field.

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} lesson pages")

Loaded 72 lesson pages


## Q1. Embedding a query

An **embedding** turns text into a fixed-length array of numbers. The model is trained so
that texts with similar meaning produce vectors that point in similar directions.

We use the `Embedder` class from `embedder.py`. Under the hood it:
1. Tokenizes the text
2. Runs it through the ONNX model
3. Averages the token vectors (mean pooling)
4. Normalizes the result to unit length (norm = 1)

The normalization step is key: when vectors have norm = 1, the dot product between them
equals their cosine similarity. No extra formula needed.

In [4]:
embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(f"Vector shape: {v.shape}")
print(f"Vector norm:  {np.linalg.norm(v):.4f}  ← should be 1.0 (normalized)")
print(f"First value v[0]: {v[0]:.4f}")

Vector shape: (384,)
Vector norm:  1.0000  ← should be 1.0 (normalized)
First value v[0]: -0.0206


**Answer Q1: `-0.02`**

## Q2. Cosine similarity

**Cosine similarity** measures the angle between two vectors:
- 1.0 → same direction (very similar meaning)
- 0.0 → perpendicular (unrelated)
- −1.0 → opposite directions

Because our vectors are normalized (norm = 1), we can compute cosine similarity
as a simple dot product: `v1.dot(v2)` = cos(angle between them).

We'll embed the content of one specific lesson page and see how similar it is to our query.

In [5]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(d for d in documents if d["filename"] == target_filename)

v_doc = embedder.encode(target_doc["content"])

similarity = v.dot(v_doc)
print(f"Cosine similarity between query and '{target_filename}':")
print(f"  {similarity:.4f}")

Cosine similarity between query and '02-vector-search/lessons/07-sqlitesearch-vector.md':
  0.3611


**Answer Q2: `0.37`**

Makes sense: the lesson is about *using* SQLite for vector search, not about how ANN search
works internally. Some overlap, but not highly similar.

## Q3. Chunking and search by hand

A full lesson page can cover many topics, which dilutes its embedding — the vector is a
blend of all topics and matches nothing strongly.

**Chunking** splits each page into overlapping windows:
- `size=2000` characters per chunk
- `step=1000` → chunks overlap by 1000 characters so no idea falls between two windows

Each chunk keeps the original `filename` plus a `start` offset so we know where it came from.

To find the best chunk we:
1. Embed all chunks at once with `encode_batch` → a 2D matrix `X`
2. Compute `X.dot(v)` → one similarity score per chunk (matrix-vector multiplication)
3. Take `argmax` → index of the highest-scoring chunk

In [6]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}  (from {len(documents)} documents)")

Total chunks: 295  (from 72 documents)


In [7]:
# encode_batch is much faster than calling encode() in a loop
print("Encoding all chunks (this takes a moment)...")
embeddings = embedder.encode_batch([c["content"] for c in chunks])
X = np.array(embeddings)  # shape: (n_chunks, 384)
print(f"Embedding matrix shape: {X.shape}")

Encoding all chunks (this takes a moment)...
Embedding matrix shape: (295, 384)


In [8]:
# One matrix-vector multiply → one score per chunk
scores = X.dot(v)

top_idx = np.argmax(scores)
top_chunk = chunks[top_idx]

print(f"Highest-scoring chunk:")
print(f"  filename : {top_chunk['filename']}")
print(f"  start    : {top_chunk['start']}")
print(f"  score    : {scores[top_idx]:.4f}")

Highest-scoring chunk:
  filename : 02-vector-search/lessons/07-sqlitesearch-vector.md
  start    : 1000
  score    : 0.6489


**Answer Q3: `02-vector-search/lessons/07-sqlitesearch-vector.md`**

Interesting: the same file as Q2. When we search the *chunk* that covers the
ANN search section of that lesson, the similarity jumps from 0.37 (whole page)
to 0.65 (focused chunk) — chunking helps.

## Q4. Vector search with minsearch

Doing the dot product manually is instructive but not practical. `minsearch.VectorSearch`
encapsulates the same logic in a library.

**Key difference from `minsearch.Index` (keyword search):**
- `VectorSearch.fit(vectors, payload)` takes a **pre-computed matrix** of vectors
- `VectorSearch.search(query_vector, ...)` takes a **pre-computed query vector**

We reuse the `X` matrix and `embedder` from Q3 — no need to re-encode everything.

In [9]:
vs = minsearch.VectorSearch(keyword_fields=["filename"])
vs.fit(X, chunks)  # X: embedding matrix, chunks: the documents

query_q4 = "What metric do we use to evaluate a search engine?"
v_q4 = embedder.encode(query_q4)

results_q4 = vs.search(v_q4, num_results=5)

print("Top 5 results:")
for i, r in enumerate(results_q4):
    print(f"  {i+1}. {r['filename']}")

Top 5 results:
  1. 04-evaluation/lessons/05-search-metrics.md
  2. 04-evaluation/lessons/01-intro.md
  3. 01-agentic-rag/lessons/05-search.md
  4. 04-evaluation/lessons/01-intro.md
  5. 04-evaluation/lessons/15-next-steps.md


**Answer Q4: `04-evaluation/lessons/05-search-metrics.md`**

Notice that vector search found a lesson from a *different module* (module 04: Evaluation),
even though we indexed lessons from all modules. It matched by meaning — the query is about
evaluating search, and that lesson is exactly about search metrics.

## Q5. Text search vs vector search

Vector and keyword search find different things:
- **Keyword search** matches exact words → finds documents that *contain the words*
- **Vector search** matches meaning → finds documents that *mean the same thing*

For 'How do I store vectors in PostgreSQL?' a keyword engine will match anything with
'vectors' or 'PostgreSQL'. A vector engine will also find pages that talk about PGVector
without using those exact words.

We build a keyword index with `minsearch.Index` on the same chunks, run both searches,
and compare the unique filenames in each top-5.

In [10]:
ki = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
ki.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"
v_q5 = embedder.encode(query_q5)

kw_results = ki.search(query_q5, num_results=5)
vec_results = vs.search(v_q5, num_results=5)

kw_files = {r["filename"] for r in kw_results}
vec_files = {r["filename"] for r in vec_results}

print("Keyword results (unique files):")
for f in sorted(kw_files):
    print(f"  {f}")

print("\nVector results (unique files):")
for f in sorted(vec_files):
    print(f"  {f}")

only_in_vector = vec_files - kw_files
print(f"\nIn vector but NOT in keyword: {only_in_vector}")

Keyword results (unique files):
  02-vector-search/lessons/01-intro.md
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md

Vector results (unique files):
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md

In vector but NOT in keyword: {'02-vector-search/lessons/08-pgvector.md'}


**Answer Q5: `02-vector-search/lessons/08-pgvector.md`**

The PGVector lesson appears in vector results but not in keyword results — because vector
search recognized that it's about storing vectors in PostgreSQL even without exact keyword overlap.

## Q6. Hybrid search with Reciprocal Rank Fusion

Neither search method is universally better. We can get the best of both worlds with
**hybrid search**, which combines their ranked lists.

**Reciprocal Rank Fusion (RRF)** ignores the raw scores (they live on different scales)
and looks only at the rank position of each document in each list:

```
RRF score = Σ  1 / (k + rank)
```

Where `rank` starts at 0, and `k=60` is a constant that dampens the effect of very
top-ranked results. Documents that rank high in *multiple* lists get a high combined score
— even if they're not #1 in any single list.

In [11]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [12]:
query_q6 = "How do I give the model access to tools?"
v_q6 = embedder.encode(query_q6)

kw_q6 = ki.search(query_q6, num_results=5)
vec_q6 = vs.search(v_q6, num_results=5)
rrf_results = rrf([kw_q6, vec_q6])

print(f"Keyword top-1 : {kw_q6[0]['filename']}")
print(f"Vector  top-1 : {vec_q6[0]['filename']}")
print(f"RRF     top-1 : {rrf_results[0]['filename']}")
print()
print("RRF top 5:")
for i, r in enumerate(rrf_results):
    print(f"  {i+1}. {r['filename']}")

Keyword top-1 : 01-agentic-rag/lessons/14-agentic-loop.md
Vector  top-1 : 01-agentic-rag/lessons/01-intro.md
RRF     top-1 : 01-agentic-rag/lessons/13-function-calling.md

RRF top 5:
  1. 01-agentic-rag/lessons/13-function-calling.md
  2. 01-agentic-rag/lessons/14-agentic-loop.md
  3. 01-agentic-rag/lessons/01-intro.md
  4. 04-evaluation/lessons/02-ground-truth.md
  5. 01-agentic-rag/lessons/13-function-calling.md


**Answer Q6: `01-agentic-rag/lessons/13-function-calling.md`**

This file wins RRF even though it's not #1 in either individual search — it ranks
consistently high in *both* lists, and RRF rewards that consistency.

This is the core intuition of hybrid search: a document that is "pretty good" across
multiple methods often beats a document that is "great" in only one.

## Summary of answers

| Question | Answer |
|---|---|
| Q1. First value of embedding vector `v[0]` | `-0.02` |
| Q2. Cosine similarity with `07-sqlitesearch-vector.md` | `0.37` |
| Q3. File with highest-scoring chunk (ANN query) | `02-vector-search/lessons/07-sqlitesearch-vector.md` |
| Q4. First minsearch VectorSearch result (evaluation query) | `04-evaluation/lessons/05-search-metrics.md` |
| Q5. File in vector results but not in keyword results | `02-vector-search/lessons/08-pgvector.md` |
| Q6. First result after RRF hybrid search | `01-agentic-rag/lessons/13-function-calling.md` |

Submit at: https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw2